In [1]:
import numpy as np
import os
import urllib.request

# 1. define target classes
# We'll start with 5 core classes
classes = ['drums', 'sun', 'laptop', 'hat', 'tree']

# Create a folder in Colab to store the files
os.makedirs('quickdraw_data', exist_ok=True)
base_url = 'https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/'

# 2. download channels
print("⏳ Downloading QuickDraw .npy files from cloud storage...")
for cl in classes:
    cls_url = cl.replace(' ', '%20')
    path = f"quickdraw_data/{cl}.npy"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"{base_url}{cls_url}.npy", path)
        print(f"📦 Retrieved: {cl}.npy")

# memory saving data loading
X_list = []
y_list = []

print("\n⏳ Slicing memory-mapped arrays (5,000 samples per class)...")
for idx, cl in enumerate(classes):
    path = f"quickdraw_data/{cl}.npy"

    # mmap_mode='r' reads the file without loading it into RAM
    data = np.load(path, mmap_mode='r')

    # Pull out exactly the first 5,000 rows into memory
    sampled_data = np.array(data[:5000])

    X_list.append(sampled_data)
    y_list.append(np.full(5000, idx)) # Assign a label number (0 to 4)
    print(f"Class '{cl}' loaded successfully into memory. Shape: {sampled_data.shape}")

# Combine everything into unified training matrices
X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

print(f"\nFinal arrays compiled safely without crashing!")
print(f"Total Images Matrix Shape: {X.shape} (RAM usage is perfectly stable)")
print(f"Total Labels Matrix Shape: {y.shape}")

⏳ Downloading QuickDraw .npy files from cloud storage...
📦 Retrieved: drums.npy
📦 Retrieved: sun.npy
📦 Retrieved: laptop.npy
📦 Retrieved: hat.npy
📦 Retrieved: tree.npy

⏳ Slicing memory-mapped arrays (5,000 samples per class)...
Class 'drums' loaded successfully into memory. Shape: (5000, 784)
Class 'sun' loaded successfully into memory. Shape: (5000, 784)
Class 'laptop' loaded successfully into memory. Shape: (5000, 784)
Class 'hat' loaded successfully into memory. Shape: (5000, 784)
Class 'tree' loaded successfully into memory. Shape: (5000, 784)

Final arrays compiled safely without crashing!
Total Images Matrix Shape: (25000, 784) (RAM usage is perfectly stable)
Total Labels Matrix Shape: (25000,)


In [2]:
# image reshaping, normalization and dataset splitting
from sklearn.model_selection import train_test_split

print("Reshaping flat 784-row pixel vectors into 28x28 2D spatial grids...")
# 1. reshape to 2D spatial tensors
# This changes the data shape from (25000, 784) into (25000, 28, 28, 1)
X_reshaped = X.reshape(-1, 28, 28, 1)

print("Normalizing pixel intensities from [0 to 255] down to bounds of [0.0 to 1.0]...")
# 2. intensity normalization
# Dividing by 255.0 converts integer pixel colors into smooth decimal floating points
X_normalized = X_reshaped.astype('float32') / 255.0

print("Executing stratified train-test dataset split (80% Train, 20% Validation)...")
# 3. dataset splitting
# We split the data so we have a completely unseen set to test our validation accuracy later
X_train, X_val, y_train, y_val = train_test_split(
    X_normalized,
    y,
    test_size=0.2,      # 20% reserved for model verification/validation
    random_state=42,    # Ensures reproducible splits every time you run it
    stratify=y          # Guarantees an equal balance of every sketch class in train and val sets
)

print("\nData Preprocessing Completed Successfully!")
print(f"Training Images Data Shape: {X_train.shape} (20,000 sketches)")
print(f"Validation Images Data Shape: {X_val.shape} (5,000 sketches)")
print(f"Target Training Labels Shape: {y_train.shape}")

Reshaping flat 784-row pixel vectors into 28x28 2D spatial grids...
Normalizing pixel intensities from [0 to 255] down to bounds of [0.0 to 1.0]...
Executing stratified train-test dataset split (80% Train, 20% Validation)...

Data Preprocessing Completed Successfully!
Training Images Data Shape: (20000, 28, 28, 1) (20,000 sketches)
Validation Images Data Shape: (5000, 28, 28, 1) (5,000 sketches)
Target Training Labels Shape: (20000,)


In [3]:
# cnn architecture
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

print("Assembling neural network layers")

# 1. initialize sequential container
model = Sequential([
    # Convolutional Block 1: 30 Filters, 3x3 sliding window, processing (28, 28, 1) inputs
    Conv2D(30, (3, 3), activation='relu', input_shape=(28, 28, 1), name="Conv_Block_1"),
    MaxPooling2D((2, 2), name="MaxPool_1"),

    # Convolutional Block 2: 15 Filters, 3x3 sliding window
    Conv2D(15, (3, 3), activation='relu', name="Conv_Block_2"),
    MaxPooling2D((2, 2), name="MaxPool_2"),

    # Overfitting Shield: Randomly drops 20% of node weights during training passes
    Dropout(0.2, name="Overfitting_Guard_Dropout"),

    # Classification Dense Blocks
    Flatten(name="Vector_Flatten"),
    Dense(128, activation='relu', name="Dense_Hidden_1"),
    Dense(50, activation='relu', name="Dense_Hidden_2"),

    # Softmax Output Layer (Set dynamically to match our active number of classes)
    Dense(len(classes), activation='softmax', name="Softmax_Output")
])


model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Network Compiled Successfully!\n")

# Print out the structural parameters to verify our layers line up perfectly
model.summary()

Assembling neural network layers


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Network Compiled Successfully!



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Conv_Block_1 (Conv2D)           │ (None, 26, 26, 30)     │           300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool_1 (MaxPooling2D)        │ (None, 13, 13, 30)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv_Block_2 (Conv2D)           │ (None, 11, 11, 15)     │         4,065 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool_2 (MaxPooling2D)        │ (None, 5, 5, 15)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Overfitting_Guard_Dropout       │ (None, 5, 5, 15)       │             0 │
│ (Dropout)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Vector_Flatten (Flatten)        │ (None, 375)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_Hidden_1 (Dense)          │ (None, 128)            │        48,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_Hidden_2 (Dense)          │ (None, 50)             │         6,450 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Softmax_Output (Dense)          │ (None, 5)              │           255 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 59,198 (231.24 KB)

 Trainable params: 59,198 (231.24 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# early stopping and gpu training
from tensorflow.keras.callbacks import EarlyStopping

print("Initializing automated Early Stopping regularization guard...")
# early stopping protocols
# This monitors validation loss, allows a 'patience' window of 3 epochs of no improvement,
# and restores the absolute best weight settings once stopped.
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("Launching model training loop.")
# trigger fit loop
# We will set a ceiling of 30 epochs, processing data in batches of 64 images.
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)

print("\nTraining session complete!")

Initializing automated Early Stopping regularization guard...
Launching model training loop.
Epoch 1/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.8324 - loss: 0.5153 - val_accuracy: 0.9118 - val_loss: 0.2902
Epoch 2/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9165 - loss: 0.2639 - val_accuracy: 0.9344 - val_loss: 0.2171
Epoch 3/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9318 - loss: 0.2122 - val_accuracy: 0.9358 - val_loss: 0.2008
Epoch 4/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9406 - loss: 0.1842 - val_accuracy: 0.9432 - val_loss: 0.1690
Epoch 5/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9478 - loss: 0.1647 - val_accuracy: 0.9452 - val_loss: 0.1592
Epoch 6/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9502 - loss: 0.1512 - val_accuracy: 0.9464 - val_loss: 0.1591
Epoch 7/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9541 - loss: 0.1405 - val_accuracy: 0.9480 - val_loss: 0.1539
Epoch 8/3

In [5]:
# export trained weights
# Save the model weights and architecture into a single, unified HDF5 file
model.save("sketchxai_matching_model.h5")
print("Success: Model saved locally in Colab as 'sketchxai_matching_model.h5'")

Success: Model saved locally in Colab as 'sketchxai_matching_model.h5'


In [6]:
import os
import urllib.request

# 1. The full, curated list of your 87 classes
categories = [
    # Original Test Subsets
    'drums', 'sun', 'laptop', 'hat', 'tree',

    # Common Animals & Nature
    'ant', 'bear', 'bee', 'bird', 'butterfly', 'camel', 'cat', 'cow', 'crab', 'crocodile',
    'dog', 'dolphin', 'dragon', 'duck', 'elephant', 'fish', 'frog', 'giraffe', 'hedgehog', 'horse',
    'lion', 'monkey', 'mouse', 'octopus', 'owl', 'penguin', 'pig', 'rabbit', 'shark', 'sheep',
    'snail', 'snake', 'spider', 'squirrel', 'tiger', 'whale', 'zebra', 'flower', 'mushroom',

    # Everyday Objects & Tools
    'alarm_clock', 'axe', 'backpack', 'banana', 'bandage', 'baseball_bat', 'bed', 'bicycle',
    'binoculars', 'book', 'bottle', 'bowtie', 'bucket', 'camera', 'candle', 'car', 'chair',
    'clock', 'cloud', 'computer', 'cup', 'door', 'envelope', 'eyeglasses', 'fan', 'fork', 'guitar',
    'hammer', 'headphones', 'house', 'ice_cream', 'key', 'knife', 'ladder', 'leaf', 'light_bulb',
    'lightning', 'moon', 'mountain', 'mug', 'pants', 'paper_clip', 'parachute', 'umbrella'
]

# 2. Setup a temporary folder inside Colab's local drive
local_download_dir = '/content/quickdraw_data'
os.makedirs(local_download_dir, exist_ok=True)

base_url = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

print(f"🔄 Starting direct download of {len(categories)} classes into temporary Colab memory...")

# 3. Download loop
for index, category in enumerate(categories):
    encoded_category = category.replace('_', '%20')
    file_name = f"{category}.npy"
    url = f"{base_url}{encoded_category}.npy"
    save_path = os.path.join(local_download_dir, file_name)

    if not os.path.exists(save_path):
        try:
            print(f"📥 [{index+1}/{len(categories)}] Downloading {file_name}...")
            urllib.request.urlretrieve(url, save_path)
        except Exception as e:
            print(f"Could not download {category}: {e}")
    else:
        print(f"{file_name} already present.")

print("\nAll 87 classes downloaded directly into Colab's local drive")

🔄 Starting direct download of 88 classes into temporary Colab memory...
drums.npy already present.
sun.npy already present.
laptop.npy already present.
hat.npy already present.
tree.npy already present.
📥 [6/88] Downloading ant.npy...
📥 [7/88] Downloading bear.npy...
📥 [8/88] Downloading bee.npy...
📥 [9/88] Downloading bird.npy...
📥 [10/88] Downloading butterfly.npy...
📥 [11/88] Downloading camel.npy...
📥 [12/88] Downloading cat.npy...
📥 [13/88] Downloading cow.npy...
📥 [14/88] Downloading crab.npy...
📥 [15/88] Downloading crocodile.npy...
📥 [16/88] Downloading dog.npy...
📥 [17/88] Downloading dolphin.npy...
📥 [18/88] Downloading dragon.npy...
📥 [19/88] Downloading duck.npy...
📥 [20/88] Downloading elephant.npy...
📥 [21/88] Downloading fish.npy...
📥 [22/88] Downloading frog.npy...
📥 [23/88] Downloading giraffe.npy...
📥 [24/88] Downloading hedgehog.npy...
📥 [25/88] Downloading horse.npy...
📥 [26/88] Downloading lion.npy...
📥 [27/88] Downloading monkey.npy...
📥 [28/88] Downloading mouse.

In [7]:
import urllib.request
import os

try:
    print("Downloading missing asset via corrected URL target...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/wine%20bottle.npy",
        "/content/quickdraw_data/bottle.npy"
    )
    print("Success! Saved as 'bottle.npy'")
except Exception as e:
    print(f"Error patching asset: {e}")

Success! Saved as 'bottle.npy'


In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# 1. VERIFY LABELS ROSTER
categories = [
    'drums', 'sun', 'laptop', 'hat', 'tree',
    'ant', 'bear', 'bee', 'bird', 'butterfly', 'camel', 'cat', 'cow', 'crab', 'crocodile',
    'dog', 'dolphin', 'dragon', 'duck', 'elephant', 'fish', 'frog', 'giraffe', 'hedgehog', 'horse',
    'lion', 'monkey', 'mouse', 'octopus', 'owl', 'penguin', 'pig', 'rabbit', 'shark', 'sheep',
    'snail', 'snake', 'spider', 'squirrel', 'tiger', 'whale', 'zebra', 'flower', 'mushroom',
    'alarm_clock', 'axe', 'backpack', 'banana', 'bandage', 'baseball_bat', 'bed', 'bicycle',
    'binoculars', 'book', 'bottle', 'bowtie', 'bucket', 'camera', 'candle', 'car', 'chair',
    'clock', 'cloud', 'computer', 'cup', 'door', 'envelope', 'eyeglasses', 'fan', 'fork', 'guitar',
    'hammer', 'headphones', 'house', 'ice_cream', 'key', 'knife', 'ladder', 'leaf', 'light_bulb',
    'lightning', 'moon', 'mountain', 'mug', 'pants', 'paper_clip', 'parachute', 'umbrella'
]

num_classes = len(categories)
samples_per_class = 5000
local_download_dir = '/content/quickdraw_data'

# 2. CREATE A SHUFFLED INDEX MAP TO PREVENT CATASTROPHIC FORGETTING
print("🎲 Mapping and shuffling cross-class index configurations...")
train_samples_per_class = int(samples_per_class * 0.75)
val_samples_per_class = samples_per_class - train_samples_per_class

train_meta = []
val_meta = []

for class_idx, category in enumerate(categories):
    for s_idx in range(train_samples_per_class):
        train_meta.append((class_idx, s_idx))
    for s_idx in range(train_samples_per_class, samples_per_class):
        val_meta.append((class_idx, s_idx))

# Shuffle them completely out of order so every batch contains a dynamic mix of all shapes
train_meta = np.array(train_meta)
val_meta = np.array(val_meta)
np.random.shuffle(train_meta)
np.random.shuffle(val_meta)

# Memory-map the open disk pointers instantly
mmaps = {
    idx: np.load(os.path.join(local_download_dir, f"{cat}.npy"), mmap_mode='r')
    for idx, cat in enumerate(categories)
}

# 3. GLOBAL SHUFFLED DATA GENERATOR
def shuffled_generator(meta_records, mmap_dict, batch_size=128):
    while True:
        # Re-shuffle indices at the start of every epoch loop
        indices = np.arange(len(meta_records))
        np.random.shuffle(indices)

        X_batch, y_batch = [], []
        for idx in indices:
            class_idx, sample_idx = meta_records[idx]

            # Fetch single frame from disk instantly via file lookup pointer
            sample = mmap_dict[class_idx][sample_idx]

            X_batch.append(sample.reshape(28, 28, 1).astype('float32') / 255.0)
            y_batch.append(tf.keras.utils.to_categorical(class_idx, num_classes=num_classes))

            if len(X_batch) == batch_size:
                yield np.array(X_batch), np.array(y_batch)
                X_batch, y_batch = [], []

# 4. STREAM COMPILATION
batch_size = 128
train_steps = len(train_meta) // batch_size
val_steps = len(val_meta) // batch_size

train_dataset = tf.data.Dataset.from_generator(
    lambda: shuffled_generator(train_meta, mmaps, batch_size),
    output_signature=(
        tf.TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(None, num_classes), dtype=tf.float32)
    )
).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: shuffled_generator(val_meta, mmaps, batch_size),
    output_signature=(
        tf.TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(None, num_classes), dtype=tf.float32)
    )
).prefetch(tf.data.AUTOTUNE)

print("🚀 Shuffled streaming streams compiled successfully.")

# 5. INITIALIZE NETWORK MODEL
model = Sequential([
    Conv2D(30, (3, 3), activation='relu', input_shape=(28, 28, 1), name="Conv_Block_1"),
    MaxPooling2D((2, 2), name="MaxPool_1"),
    Conv2D(15, (3, 3), activation='relu', name="Conv_Block_2"),
    MaxPooling2D((2, 2), name="MaxPool_2"),
    Dropout(0.2, name="Overfitting_Guard_Dropout"),
    Flatten(name="Vector_Flatten"),
    Dense(128, activation='relu', name="Dense_Hidden_1"),
    Dense(50, activation='relu', name="Dense_Hidden_2"),
    Dense(num_classes, activation='softmax', name="Softmax_Output")
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint = ModelCheckpoint(
    "sketchxai_matching_model.h5",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# 6. RUN BALANCED TRAINING
print("🏋️ Starting Fully Shuffled Training Pass...")
model.fit(
    train_dataset,
    steps_per_epoch=train_steps,
    validation_data=val_dataset,
    validation_steps=val_steps,
    epochs=12,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)

🎲 Mapping and shuffling cross-class index configurations...
🚀 Shuffled streaming streams compiled successfully.
🏋️ Starting Fully Shuffled Training Pass...
Epoch 1/12
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3582 - loss: 2.6732
Epoch 1: val_accuracy improved from None to 0.60848, saving model to sketchxai_matching_model.h5



Epoch 1: finished saving model to sketchxai_matching_model.h5
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - accuracy: 0.4799 - loss: 2.1067 - val_accuracy: 0.6085 - val_loss: 1.5367
Epoch 2/12
2576/2578 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5996 - loss: 1.5734
Epoch 2: val_accuracy improved from 0.60848 to 0.66327, saving model to sketchxai_matching_model.h5



Epoch 2: finished saving model to sketchxai_matching_model.h5
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 31s 12ms/step - accuracy: 0.6110 - loss: 1.5277 - val_accuracy: 0.6633 - val_loss: 1.3189
Epoch 3/12
2577/2578 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6380 - loss: 1.4092
Epoch 3: val_accuracy improved from 0.66327 to 0.67895, saving model to sketchxai_matching_model.h5



Epoch 3: finished saving model to sketchxai_matching_model.h5
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 31s 12ms/step - accuracy: 0.6426 - loss: 1.3919 - val_accuracy: 0.6790 - val_loss: 1.2549
Epoch 4/12
2575/2578 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6573 - loss: 1.3262
Epoch 4: val_accuracy improved from 0.67895 to 0.69825, saving model to sketchxai_matching_model.h5



Epoch 4: finished saving model to sketchxai_matching_model.h5
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 31s 12ms/step - accuracy: 0.6598 - loss: 1.3159 - val_accuracy: 0.6983 - val_loss: 1.1796
Epoch 5/12
2576/2578 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6714 - loss: 1.2700
Epoch 5: val_accuracy improved from 0.69825 to 0.70572, saving model to sketchxai_matching_model.h5


...


Epoch 8: finished saving model to sketchxai_matching_model.h5
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 37s 14ms/step - accuracy: 0.6916 - loss: 1.1818 - val_accuracy: 0.7193 - val_loss: 1.0838
Epoch 9/12
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6963 - loss: 1.1641
Epoch 9: val_accuracy improved from 0.71926 to 0.72045, saving model to sketchxai_matching_model.h5



Epoch 9: finished saving model to sketchxai_matching_model.h5
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 39s 15ms/step - accuracy: 0.6956 - loss: 1.1650 - val_accuracy: 0.7205 - val_loss: 1.0743
Epoch 10/12
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6989 - loss: 1.1505
Epoch 10: val_accuracy improved from 0.72045 to 0.72423, saving model to sketchxai_matching_model.h5



Epoch 10: finished saving model to sketchxai_matching_model.h5
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 37s 14ms/step - accuracy: 0.6990 - loss: 1.1507 - val_accuracy: 0.7242 - val_loss: 1.0619
Epoch 11/12
2575/2578 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7034 - loss: 1.1326
Epoch 11: val_accuracy improved from 0.72423 to 0.72632, saving model to sketchxai_matching_model.h5



Epoch 11: finished saving model to sketchxai_matching_model.h5
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 37s 14ms/step - accuracy: 0.7024 - loss: 1.1348 - val_accuracy: 0.7263 - val_loss: 1.0511
Epoch 12/12
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7059 - loss: 1.1216
Epoch 12: val_accuracy improved from 0.72632 to 0.72680, saving model to sketchxai_matching_model.h5



Epoch 12: finished saving model to sketchxai_matching_model.h5
2578/2578 ━━━━━━━━━━━━━━━━━━━━ 38s 15ms/step - accuracy: 0.7055 - loss: 1.1244 - val_accuracy: 0.7268 - val_loss: 1.0458
Restoring model weights from the end of the best epoch: 12.
